# Process data

Joins the aggregated responses with the student data, adds the columns the dashboard groups by,
and replaces student numbers with a pseudonymous `visitor_id`.

| Table | Role |
|---|---|
| `processed_data.aggregated_responses` | input, from `aggregated_customer_data.ipynb` |
| `static_data.students` | input, from `update_static_data.ipynb` |
| `processed_data.processed_responses` | output, **replaced** on every run |

The output keeps only the columns listed in `COLUMNS`, so it has no student numbers, usernames or
names, and a new column in the source stays out until it is added there on purpose. `visitor_id` is
pseudonymous, not anonymous: anyone who can read `aggregated_responses` can map it back to student
numbers, and rare combinations of the student data can single someone out. Share the output and the
dashboard only with people who may see the survey data.

In [ ]:
from pis_lib.connect import get_pg_engine
import pandas as pd

In [ ]:
pg = get_pg_engine()

## Load

- A few student numbers appear twice in `students`; keep the first row so the join can't duplicate responses.
- `workshop` and `used_ai` become pandas' `boolean` type (true/false with `<NA>`); read from the database, true/false
  columns with missing values come back as `object`.

In [ ]:
responses_df = pd.read_sql("SELECT * FROM processed_data.aggregated_responses", pg).astype(
    {"workshop": "boolean", "used_ai": "boolean"}
)
students_df = pd.read_sql(
    'SELECT student_number, gender, "education_EN", "direction_DK", "direction_EN" FROM static_data.students', pg
).drop_duplicates("student_number")

## Join and derive columns

- **Student data**: `gender`, `education_EN`, `direction_DK`, `direction_EN`. NULL when the response has no
  student number or the student isn't in `students`.
- **Times** use Copenhagen wall-clock time:
  - `weekday`: `Monday` … `Sunday`.
  - `time_slot`: `Morning` before 12:30, else `Afternoon`.
  - `semester`: `Spring 25` for January to August, `Fall 25` for September to December.
- **`study_start`**: for students found in `students`, `s25` from the year digits of the student number, or
  `s22 and older` for those who started before 2023. NULL for everyone else.
- **`verified`**: the student number is in `students`.
- **`visitor_id`**: one number per student number, NULL for anonymous responses. Unique visitor counts use it.
- **`staff`**: the response gave staff initials (`username`) instead of a student number. Only this flag is
  kept, not the initials.

In [ ]:
local_time = responses_df["created_at"].dt.tz_convert("Europe/Copenhagen")
start_year = responses_df["student_number"].str[1:3]
recent_start = pd.to_numeric(start_year, errors="coerce").between(23, pd.Timestamp.now().year % 100)  # s99 is 1999
known_student = responses_df["student_number"].isin(students_df["student_number"])
codes = pd.factorize(responses_df["student_number"])[0]  # -1 where there is no student number
COLUMNS = [
    "created_at", "satisfaction", "course_number", "building_number", "workshop", "used_ai",
    "gender", "education_EN", "direction_DK", "direction_EN",
    "weekday", "time_slot", "semester", "study_start", "verified", "visitor_id", "staff",
]

processed_df = (
    responses_df
    .merge(students_df, on="student_number", how="left", validate="many_to_one")
    .assign(
        weekday=local_time.dt.day_name(),
        time_slot=(local_time.dt.strftime("%H:%M") < "12:30").map({True: "Morning", False: "Afternoon"}),
        semester=(local_time.dt.month <= 8).map({True: "Spring ", False: "Fall "}) + local_time.dt.strftime("%y"),
        study_start=("s" + start_year).where(recent_start, "s22 and older").where(known_student),
        verified=known_student,
        visitor_id=pd.Series(codes, index=responses_df.index).where(codes >= 0).astype("Int64"),
        staff=responses_df["username"].notna(),
    )
    [COLUMNS]
)

assert len(processed_df) == len(responses_df)
assert processed_df["visitor_id"].nunique() == responses_df["student_number"].nunique()

## Write to `processed_data`

> **Warning:** `if_exists="replace"` drops and recreates the table on every run.

In [ ]:
processed_df.to_sql("processed_responses", pg, schema="processed_data", if_exists="replace", index=False)